# Custom Layer Implementation
* Task: Do not use nn.Linear. Instead, build a custom "Linear Layer" class by explicitly defining the Weight and Bias parameters using
* torch.nn.Parameter. Implement "Xavier/Glorot Initialization" manually for your weights. Add this custom layer into a larger network.

Why: Shows you that "layers" are just wrappers around raw tensors with registered parameters that the optimizer knows to update.


In [1]:
import math
import torch
import torch.nn as nn
class CustomLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(CustomLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # 1. Allocate raw tensors for weights and biases
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))
        
        # 2. Apply manual Xavier/Glorot Initialization
        self.init_weights()
        
    def init_weights(self):
        # Calculate Xavier bounds
        bound = math.sqrt(6.0 / (self.in_features + self.out_features))
        
        # Fill the parameters uniformly in-place
        nn.init.uniform_(self.weight, -bound, bound)
        nn.init.zeros_(self.bias) # Biases are typically initialized to zero
        
    def forward(self, x):
        # Implement the standard linear transformation: y = xW^T + b
        return torch.matmul(x, self.weight.t()) + self.bias
class CustomMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CustomMLP, self).__init__()
        
        # Stack our custom layers alongside native activation functions
        self.layer1 = CustomLinear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = CustomLinear(hidden_size, output_size)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out
# Instantiate the model
model = CustomMLP(input_size=20, hidden_size=64, output_size=5)

# 1. Verify parameters are registered
print("Registered Model Parameters:")
for name, param in model.named_parameters():
    print(f"-> {name}: {param.shape}")

print("-" * 40)

# 2. Test forward pass with dummy data (batch_size=4, features=20)
dummy_input = torch.randn(4, 20)
dummy_output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")   # Expected: torch.Size([4, 20])
print(f"Output shape: {dummy_output.shape}")  # Expected: torch.Size([4, 5])

Registered Model Parameters:
-> layer1.weight: torch.Size([64, 20])
-> layer1.bias: torch.Size([64])
-> layer2.weight: torch.Size([5, 64])
-> layer2.bias: torch.Size([5])
----------------------------------------
Input shape:  torch.Size([4, 20])
Output shape: torch.Size([4, 5])
